In [1]:
import math

import numpy as np
import scipy
import torch
from sentence_transformers import SentenceTransformer

d:\Learning\AI-Agents\my_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Example documents
documents = [
    'Bugs introduced by the intern had to be squashed by the lead developer.',
    'Bugs found by the quality assurance engineer were difficult to debug.',
    'Bugs are common throughout the warm summer months, according to the entomologist.',
    'Bugs, in particular spiders, are extensively studied by arachnologists.'
]


In [3]:
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2780.53it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# Generate embeddings
embeddings = model.encode(documents)

In [5]:
embeddings.shape

(4, 384)

In [6]:
embeddings

array([[-0.2280436 , -0.24647677, -0.00319288, ...,  0.4552815 ,
         0.6341977 ,  0.53750485],
       [-0.35791588, -0.3208399 ,  0.15963264, ..., -0.0705065 ,
         0.9275023 ,  0.34377307],
       [ 0.20302957, -0.26898596,  0.1628514 , ..., -0.19651002,
        -0.03379865,  0.59561515],
       [-0.04264297, -0.4572162 , -0.09526487, ..., -0.5803074 ,
         0.17248404,  0.09127858]], shape=(4, 384), dtype=float32)

### L2 Distance Manual

$$ \text{L2}(a,b) = \ \sqrt{\sum_{i=1}^n (a_i - b_i)^2} $$

In [7]:
#L2 Distance or Euclidean distance

def euclidean_distance_fn(vector1, vector2):
    squared_sum = sum((x - y) ** 2 for x, y in zip(vector1, vector2))
    return math.sqrt(squared_sum)

In [8]:
euclidean_distance_fn(embeddings[0], embeddings[1])

5.9617895314850395

In [9]:
euclidean_distance_fn(embeddings[1], embeddings[2])


7.768614533180078

In [11]:
l2_dist_manual = np.zeros([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        if j > i: # Calculate the upper triangle only
            l2_dist_manual[i,j] = euclidean_distance_fn(embeddings[i], embeddings[j])
        elif i > j: # Copy the uper triangle to the lower triangle
            l2_dist_manual[i,j] = l2_dist_manual[j,i]

l2_dist_manual

array([[0.        , 5.96178953, 7.33939911, 7.15578329],
       [5.96178953, 0.        , 7.76861453, 7.39359048],
       [7.33939911, 7.76861453, 0.        , 5.91992896],
       [7.15578329, 7.39359048, 5.91992896, 0.        ]])

### L2 Distance Scipy

In [12]:
l2_dist_scipy = scipy.spatial.distance.cdist(embeddings, embeddings, 'euclidean')
l2_dist_scipy

array([[0.        , 5.96178969, 7.33940022, 7.15578238],
       [5.96178969, 0.        , 7.76861568, 7.39359029],
       [7.33940022, 7.76861568, 0.        , 5.91992812],
       [7.15578238, 7.39359029, 5.91992812, 0.        ]])

In [13]:
np.allclose(l2_dist_manual, l2_dist_scipy)

True

### Dot Product Similarity and Distance

$$ a \cdot b = \sum_{i=1}^{n} a_i b_i $$

In [14]:
def dot_product_fn(vector1, vector2):
    return sum(x * y for x, y in zip(vector1, vector2))

dot_product_fn(embeddings[0], embeddings[1])

np.float32(18.535395)

In [15]:
dot_product_manual = np.empty([4,4])
for i in range(embeddings.shape[0]):
    for j in range(embeddings.shape[0]):
        dot_product_manual[i,j] = dot_product_fn(embeddings[i], embeddings[j])

dot_product_manual

array([[33.74441528, 18.53539467,  8.56980896,  7.83092928],
       [18.53539467, 38.86932373,  7.88996887,  8.66340542],
       [ 8.56980896,  7.88996887, 37.26200867, 17.66956139],
       [ 7.83092928,  8.66340542, 17.66956139, 33.12266159]])

### Calculate the dot product using matrix multiplication

In [16]:
# Matrix multiplication operator
dot_product_operator = embeddings @ embeddings.T
dot_product_operator

array([[33.744415, 18.535395,  8.569809,  7.830929],
       [18.535395, 38.869324,  7.889969,  8.663404],
       [ 8.569809,  7.889969, 37.26201 , 17.669561],
       [ 7.830929,  8.663404, 17.669561, 33.12266 ]], dtype=float32)

In [17]:
np.allclose(dot_product_manual, dot_product_operator, atol=1e-05)

True

In [18]:
# Equivalent to `np.matmul()` if both arrays are 2-D:
np.matmul(embeddings,embeddings.T)

# `np.dot` returns an identical result, but `np.matmul` is recommended if both arrays are 2-D:
np.dot(embeddings,embeddings.T)

array([[33.744415, 18.535395,  8.569809,  7.830929],
       [18.535395, 38.869324,  7.889969,  8.663404],
       [ 8.569809,  7.889969, 37.26201 , 17.669561],
       [ 7.830929,  8.663404, 17.669561, 33.12266 ]], dtype=float32)

### Calculate dot product distance

In [19]:
dot_product_distance = -dot_product_manual
dot_product_distance

array([[-33.74441528, -18.53539467,  -8.56980896,  -7.83092928],
       [-18.53539467, -38.86932373,  -7.88996887,  -8.66340542],
       [ -8.56980896,  -7.88996887, -37.26200867, -17.66956139],
       [ -7.83092928,  -8.66340542, -17.66956139, -33.12266159]])

### The cosine similarity 
$$ \text{cossim}(a, b) = \frac{a \cdot b}{||a|| \ ||b||} =  \frac{a}{||a||} \cdot \frac{b}{||b||} $$

where $||a|| = \sqrt{\sum_{k=1}^n a_k^2}$ is the L2 norm, or the magnitude, of vector $a$, and $\cdot$ is the dot product.

Also note that $\frac{a}{||a||}$ represents a normalized vector. This means it has the same direction as vector $a$ but a magnitude (or length) of 1. Thus, cosine similarity can be calculated by taking the dot product of two normalized vectors.

### Calculate the L2 norm

In [20]:
# L2 norms
l2_norms = np.sqrt(np.sum(embeddings**2, axis=1))
l2_norms

array([5.8089943, 6.2345266, 6.104261 , 5.75523  ], dtype=float32)

In [21]:
# L2 norms reshaped
l2_norms_reshaped = l2_norms.reshape(-1,1)
l2_norms_reshaped

array([[5.8089943],
       [6.2345266],
       [6.104261 ],
       [5.75523  ]], dtype=float32)

In [22]:
normalized_embeddings_manual = embeddings/l2_norms_reshaped
normalized_embeddings_manual

array([[-0.03925699, -0.0424302 , -0.00054964, ...,  0.07837527,
         0.10917513,  0.09252976],
       [-0.05740867, -0.05146179,  0.02560461, ..., -0.01130904,
         0.14876868,  0.0551402 ],
       [ 0.0332603 , -0.04406528,  0.02667831, ..., -0.03219227,
        -0.00553689,  0.09757367],
       [-0.00740943, -0.0794436 , -0.01655275, ..., -0.10083131,
         0.02996996,  0.01586011]], shape=(4, 384), dtype=float32)

In [23]:
np.sqrt(np.sum(normalized_embeddings_manual**2, axis=1))

array([1.        , 1.        , 1.        , 0.99999994], dtype=float32)

### Normalize embeddings using PyTorch

In [24]:
normalized_embeddings_torch = torch.nn.functional.normalize(
    torch.from_numpy(embeddings)
).numpy()
normalized_embeddings_torch

array([[-0.03925699, -0.0424302 , -0.00054964, ...,  0.07837527,
         0.10917513,  0.09252976],
       [-0.05740867, -0.05146179,  0.02560461, ..., -0.01130904,
         0.14876868,  0.0551402 ],
       [ 0.0332603 , -0.04406527,  0.02667831, ..., -0.03219227,
        -0.00553689,  0.09757366],
       [-0.00740943, -0.0794436 , -0.01655275, ..., -0.10083131,
         0.02996996,  0.01586011]], shape=(4, 384), dtype=float32)

In [ ]:
np.allclose(normalized_embeddings_manual, normalized_embeddings_torch)

In [25]:
dot_product_fn(normalized_embeddings_manual[0], normalized_embeddings_manual[1])

np.float32(0.5117968)

In [26]:
cosine_similarity_manual = np.empty([4,4])
for i in range(normalized_embeddings_manual.shape[0]):
    for j in range(normalized_embeddings_manual.shape[0]):
        cosine_similarity_manual[i,j] = dot_product_fn(
            normalized_embeddings_manual[i], 
            normalized_embeddings_manual[j]
        )

cosine_similarity_manual

array([[1.        , 0.51179677, 0.24167806, 0.23423393],
       [0.51179677, 1.00000012, 0.20731878, 0.24144742],
       [0.24167806, 0.20731878, 1.00000048, 0.50295597],
       [0.23423393, 0.24144742, 0.50295597, 0.99999982]])

In [27]:
cosine_similarity_operator = normalized_embeddings_manual @ normalized_embeddings_manual.T
cosine_similarity_operator

array([[0.99999994, 0.5117968 , 0.24167807, 0.23423396],
       [0.5117968 , 1.0000001 , 0.20731877, 0.24144745],
       [0.24167807, 0.20731877, 1.0000006 , 0.5029559 ],
       [0.23423396, 0.24144745, 0.5029559 , 0.9999998 ]], dtype=float32)

In [28]:
np.allclose(cosine_similarity_manual, cosine_similarity_operator)

True

### Calculate cosine distance

$$ 1 - cossim(a,b) $$

In [29]:
1 - cosine_similarity_manual

array([[ 0.00000000e+00,  4.88203228e-01,  7.58321941e-01,
         7.65766069e-01],
       [ 4.88203228e-01, -1.19209290e-07,  7.92681217e-01,
         7.58552581e-01],
       [ 7.58321941e-01,  7.92681217e-01, -4.76837158e-07,
         4.97044027e-01],
       [ 7.65766069e-01,  7.58552581e-01,  4.97044027e-01,
         1.78813934e-07]])

In [34]:
query_embed = model.encode("Who is responsible for a coding project and fixing others' mistakes?")

query_normalize = torch.nn.functional.normalize(
    torch.from_numpy(query_embed),
    dim=0
).numpy()
query_normalize

array([-9.32223201e-02, -9.06476304e-02, -2.34147198e-02, -7.88929015e-02,
        6.19201139e-02, -2.73298495e-03,  2.70607993e-02,  3.60899866e-02,
       -1.52260503e-02,  6.16317876e-02, -6.45065531e-02,  4.59578224e-02,
        3.77347507e-02,  3.10424827e-02, -8.06073770e-02,  6.14652373e-02,
       -9.75921229e-02, -1.06532210e-02,  9.80684906e-03, -2.82161124e-02,
       -9.44458917e-02, -1.02409469e-02,  4.62454967e-02, -1.86300613e-02,
        5.65065816e-03,  5.75630739e-02,  5.76724717e-03, -2.09801886e-02,
       -3.76648013e-03, -5.10750562e-02, -4.06225696e-02,  5.92454337e-03,
        9.21309590e-02, -1.25674531e-02,  5.95897529e-03,  1.32811323e-01,
       -1.57071035e-02,  5.56269055e-03,  4.10475349e-03, -5.03916442e-02,
       -2.56171357e-02,  6.90139309e-02,  2.38441583e-02, -1.47721907e-02,
        2.20946800e-02,  1.71092171e-02,  6.69866987e-03,  7.37365801e-03,
       -6.79530427e-02, -2.79426537e-02,  4.92179729e-02,  1.30206328e-02,
       -5.10753430e-02, -

In [38]:
cosine_similarity_q3 = normalized_embeddings_manual @ query_normalize.T

# Fourth, find the position of the vector with the highest cosine similarity:
highest_cossim_position = cosine_similarity_q3.argmax()

# Fifth, find the document in that position in the `documents` array:
documents[highest_cossim_position]

'Bugs introduced by the intern had to be squashed by the lead developer.'